In [2]:
import pandas as pd
import os
from itertools import chain
import glob
import fnmatch
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from enum import Enum
from datetime import datetime
import requests
from bs4 import BeautifulSoup
import json
import re

In [3]:
class LabelType(Enum):
    MARGIN_OF_VICTORY = "mov"
    WIN_PROBABILITY = "win_probability"

class TournamentDivision(Enum):
    MEN = "M"
    WOMEN = "W"

In [4]:
SEASON = 2026
DIVISION = TournamentDivision.MEN
LABEL_TYPE = LabelType.WIN_PROBABILITY
REFRESH_KENPOM = False
EPOCHS = 100

# Data collection

In [5]:
DATA_DIRS = [f'march-machine-learning-mania-{SEASON}', '../kenpom']

CSV = {}
for path in list(chain(*map(lambda x: glob.glob(x + '/*.csv'), DATA_DIRS))):
    CSV[os.path.basename(path).split('.')[0]] = pd.read_csv(path, encoding='cp1252')
print(CSV.keys())

dict_keys(['MNCAATourneyDetailedResults', 'SampleSubmissionStage2', 'WSecondaryTourneyTeams', 'WNCAATourneySlots', 'MNCAATourneyCompactResults', 'MSeasons', 'SampleSubmissionStage1', 'WTeams', 'MRegularSeasonDetailedResults', 'WNCAATourneyDetailedResults', 'MNCAATourneySlots', 'MGameCities', 'MConferenceTourneyGames', 'WNCAATourneyCompactResults', 'WSecondaryTourneyCompactResults', 'WSeasons', 'Cities', 'WRegularSeasonCompactResults', 'WTeamSpellings', 'WRegularSeasonDetailedResults', 'MRegularSeasonCompactResults', 'WNCAATourneySeeds', 'MNCAATourneySeedRoundSlots', 'WConferenceTourneyGames', 'WTeamConferences', 'MTeamConferences', 'MTeamCoaches', 'MMasseyOrdinals', 'Conferences', 'MTeams', 'WGameCities', 'MNCAATourneySeeds', 'MSecondaryTourneyTeams', 'MTeamSpellings', 'MSecondaryTourneyCompactResults', 'kenpom_2011-2024', 'kenpom_2025-03-19', 'manual_team_spellings', 'kenpom_2025'])


In [6]:
def normalize_team_name(name: str) -> str:
    s = name.lower().strip()
    s = re.sub(r'&(\w);', r'&\1', s)  # Fix HTML artifacts: &M; -> &M
    s = s.replace('.', '').replace("'", '').replace('-', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    return s

In [7]:
manual_df = pd.read_csv('../kenpom/manual_team_spellings.csv')

mteam_spellings = CSV['MTeamSpellings']
mteam_spellings = pd.concat([mteam_spellings, manual_df], ignore_index=True)
mteam_spellings = mteam_spellings.drop_duplicates(subset=['TeamNameSpelling'], keep='first')

# Validate manual TeamIDs exist in MTeams
valid_ids = set(CSV['MTeams']['TeamID'])
invalid = manual_df[~manual_df['TeamID'].isin(valid_ids)]
assert invalid.empty, f"Manual spellings reference invalid TeamIDs: {invalid.to_dict('records')}"

teams_df = pd.merge(mteam_spellings, CSV['MTeams'], on='TeamID', how='inner').drop(['FirstD1Season', 'LastD1Season'], axis=1)

# Add normalized column for matching
teams_df['TeamNameSpelling_norm'] = teams_df['TeamNameSpelling'].apply(normalize_team_name)
teams_df = teams_df.drop_duplicates(subset=['TeamNameSpelling_norm'], keep='first')

display(teams_df)

,TeamNameSpelling,TeamID,TeamName,TeamNameSpelling_norm
0,a&m-corpus chris,1394,TAM C. Christi,a&m corpus chris
1,a&m-corpus christi,1394,TAM C. Christi,a&m corpus christi
2,abilene chr,1101,Abilene Chr,abilene chr
3,abilene christian,1101,Abilene Chr,abilene christian
5,air force,1102,Air Force,air force
...,...,...,...,...
1199,southeast missouri st.,1369,SE Missouri St,southeast missouri st
1200,southeast missouri,1369,SE Missouri St,southeast missouri
1204,texas a&m corpus chris,1394,TAM C. Christi,texas a&m corpus chris
1212,ut rio grande valley,1410,UTRGV,ut rio grande valley


## Kenpom data
### (only for Men's, n/a for Women's)

In [ ]:
# Scrape kenpom for most updated ratings in the current season
def parse_url(
    url,
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
):  
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    return soup


# Headers in the correct order as found in the html table
column_definitions = [
    {"name": "Rk", "type": "int"},
    {"name": "Team", "type": "str"},
    {"name": "Conf", "type": "str"},
    {"name": "W-L", "type": "str"},
    {"name": "NetRtg", "type": "float"},
    {"name": "ORtg", "type": "float"},
    {"name": "seed_ORtg", "type": "int"},
    {"name": "DRtg", "type": "float"},
    {"name": "seed_DRtg", "type": "int"},
    {"name": "AdjT", "type": "float"},
    {"name": "seed_AdjT", "type": "int"},
    {"name": "Luck", "type": "float"},
    {"name": "seed_Luck", "type": "int"},
    {"name": "sos_NetRtg", "type": "float"},
    {"name": "seed_sos_NetRtg", "type": "int"},
    {"name": "sos_ORtg", "type": "float"},
    {"name": "seed_sos_ORtg", "type": "int"},
    {"name": "sos_DRtg", "type": "float"},
    {"name": "seed_sos_DRtg", "type": "int"},
    {"name": "ncsos_NetRtg", "type": "float"},
    {"name": "seed_ncsos_NetRtg", "type": "int"}
]

def extract_kenpom(soup, write_to_csv = False):
    # Find the table with rankings
    table = soup.find("table", {"id": "ratings-table"})
    assert table is not None, "Error: Ratings table not found!"
    
    # Extract team data
    teams = []
    
    for row in table.find("tbody").find_all("tr"):
        cols = row.find_all("td")
    
        # assert len(cols) == len(column_definitions), f"Error: Expected {len(column_definitions)} columns but found {len(cols)} in row: {row}"
    
        if (len(cols) == len(column_definitions)):
            row_data = []
            for i, col in enumerate(cols):
                text_value = col.text.strip()
    
                # Special handling for 'Team' column (avoid capturing seed numbers)
                if column_definitions[i]["name"] == "Team":
                    team_name = col.find("a").text.strip() if col.find("a") else col.text.strip()
                    row_data.append(team_name)
                else:
                    col_type = column_definitions[i]["type"]
                    if col_type == "int":
                        row_data.append(int(text_value))
                    elif col_type == "float":
                        row_data.append(float(text_value))
                    else:
                        row_data.append(text_value)  # Keep as string for other columns
        
            teams.append(row_data)
    
    # Convert to DataFrame
    refreshed_kenpom = pd.DataFrame(teams, columns=[col["name"] for col in column_definitions])

    if (write_to_csv):
        current_date = datetime.today().strftime('%Y-%m-%d')
        refreshed_kenpom.to_csv(f'../kenpom/kenpom_{current_date}.csv', index=False)

    return refreshed_kenpom

In [ ]:
KENPOM_URL = "https://kenpom.com/"

if REFRESH_KENPOM:
    soup = parse_url(KENPOM_URL)
    refreshed_kenpom = extract_kenpom(soup, True)
else:
    # Find the most recent kenpom CSV file
    kenpom_files = sorted(
        [path for path in CSV.keys() if path.startswith("kenpom_")], 
        key=lambda x: x.split('_')[-1],  # Sort by the date in the filename
        reverse=True  # Most recent first
    )

    if kenpom_files:
        print(f'Not refreshing kenpom, using {kenpom_files[0]}')
        refreshed_kenpom = CSV[kenpom_files[0]]  # Load the most recent one
    else:
        refreshed_kenpom = None  # Handle case where no files are found


display(refreshed_kenpom)

In [ ]:
kenpom = CSV['kenpom']

kenpom['TeamName_norm'] = kenpom['TeamName'].apply(normalize_team_name)
kenpom = pd.merge(
    kenpom,
    teams_df[['TeamNameSpelling_norm', 'TeamID']],
    left_on='TeamName_norm',
    right_on='TeamNameSpelling_norm',
    how='left'
)

kenpom = kenpom.drop(['Unnamed: 0', 'TeamName_norm', 'TeamNameSpelling_norm'], axis=1)
kenpom['TeamID'] = kenpom['TeamID'].astype('Int64')

# Surface unmatched teams with actionable info
null_team_ids = kenpom[kenpom['TeamID'].isna()]
if not null_team_ids.empty:
    unmatched = null_team_ids[['TeamName']].drop_duplicates().sort_values('TeamName')
    print(f"WARNING: {len(unmatched)} team(s) could not be matched to a TeamID:")
    for _, row in unmatched.iterrows():
        norm = normalize_team_name(row['TeamName'])
        print(f"  - \"{row['TeamName']}\" (normalized: \"{norm}\")")
    print(f"\nTo fix: add entries to kenpom/manual_team_spellings.csv")
    print(f"Format: TeamNameSpelling,TeamID  (use normalized name as TeamNameSpelling)")

assert not kenpom['TeamID'].isna().any(), "Unmatched teams found — see above for details"
assert not kenpom.duplicated(subset=['Season', 'TeamID']).any(), "Duplicate 'Season' and 'TeamID' combinations found!"

In [ ]:
seasons_available = kenpom['Season'].unique()
seasons_available

# Feature engineering

In [ ]:
# Home (H) -> 1
# Away (A) -> -1
# Neutral (N) -> 0
# Flipping location can be done by multipling by -1
location_to_number_map = { 'H': 1, 'A': -1, 'N': 0 }

regular_results = CSV['MRegularSeasonCompactResults']
tourney_results = CSV['MNCAATourneyCompactResults']
assert set(regular_results.columns) == set(tourney_results.columns), "regular_results and tourney_results have different columns"
combined_results = pd.concat([regular_results, tourney_results], ignore_index=True)

available_results = combined_results[combined_results['Season'].isin(seasons_available)].copy()
available_results['WLoc_num'] = available_results['WLoc'].map(location_to_number_map)
available_results['mov'] = available_results['WScore'] - available_results['LScore']
available_results['t1_win'] = 1
display(available_results)

In [ ]:
# Rename MMasseyOrdinals as filename can be MMasseyOrdinals_something
if 'MMasseyOrdinals' not in CSV:
    CSV['MMasseyOrdinals'] = pd.concat(map(lambda x: CSV[x], fnmatch.filter(CSV.keys(), 'MMasseyOrdinals*')))

massey = CSV['MMasseyOrdinals']
available_massey = massey[massey['Season'].isin(seasons_available)]

# Group by ['Season', 'RankingDayNum', 'TeamID'] and find the average ordinal rank across all systems
real_massey = available_massey.groupby(['Season', 'RankingDayNum', 'TeamID'], as_index=False)['OrdinalRank'].mean()
assert not real_massey.duplicated(subset=['Season', 'RankingDayNum', 'TeamID']).any(), "Duplicate!"
display(real_massey.head())

In [ ]:
kenpom_columns = ['adj_o', 'adj_d', 'adj_tempo', 'luck', 'sos_adj_o', 'sos_adj_d']
team1_kenpom_columns = ['t1_' + kc for kc in kenpom_columns]
team2_kenpom_columns = ['t2_' + kc for kc in kenpom_columns]

In [ ]:
relevant_available_results = available_results[['Season', 'DayNum', 'WTeamID', 'LTeamID', 'WLoc_num', 'mov', 't1_win']]
relevant_kenpom = kenpom[['Season', 'TeamID'] + kenpom_columns]
relevant_massey = real_massey[['Season', 'RankingDayNum', 'TeamID', 'OrdinalRank']]

In [ ]:
joined_results = relevant_available_results.copy().rename(columns={'WLoc_num': 't1_loc_num'})

# add team1 (Wteam) kenpom stats
joined_results = pd.merge(joined_results, relevant_kenpom, left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID']).drop('TeamID', axis=1).rename(columns=dict(zip(kenpom_columns, team1_kenpom_columns)))

# add massey ordinal rank for team1
joined_results = pd.merge(joined_results, relevant_massey, left_on=['Season', 'DayNum', 'WTeamID'], right_on=['Season', 'RankingDayNum', 'TeamID'], how='left').drop(['TeamID', 'RankingDayNum'], axis=1).rename(columns={'OrdinalRank': 't1_OrdinalRank'})

# add team2 (Lteam) kenpom stats
joined_results = pd.merge(joined_results, relevant_kenpom, left_on=['Season', 'LTeamID'], right_on=['Season', 'TeamID']).drop('TeamID', axis=1).rename(columns=dict(zip(kenpom_columns, team2_kenpom_columns)))

# add massey ordinal rank for team2
joined_results = pd.merge(joined_results, relevant_massey, left_on=['Season', 'DayNum', 'LTeamID'], right_on=['Season', 'RankingDayNum', 'TeamID'], how='left').drop(['TeamID', 'RankingDayNum'], axis=1).rename(columns={'OrdinalRank': 't2_OrdinalRank'})

display(joined_results.tail())
print(joined_results.shape)

In [ ]:
# without massey ordinal ranks because it seems to be missing a lot of values - to add back, add 't1_OrdinalRank' and 't2_OrdinalRank'
data = joined_results[['t1_loc_num'] + team1_kenpom_columns + team2_kenpom_columns + ['mov', 't1_win']]
flipped_data = data.copy()
flipped_data[team1_kenpom_columns] = data[team2_kenpom_columns]
flipped_data[team2_kenpom_columns] = data[team1_kenpom_columns]

flipped_data['t1_loc_num'] = -data['t1_loc_num']
flipped_data['mov'] = -data['mov']
flipped_data['t1_win'] = 1 - data['t1_win']

display(data.head())
display(flipped_data.head())
print(data.shape, flipped_data.shape)

In [ ]:
augmented_data = pd.concat([data, flipped_data])
display(augmented_data)

In [ ]:
x = augmented_data.drop(['mov', 't1_win'], axis=1)
y_mov = augmented_data['mov']
y_win = augmented_data['t1_win']

# Splitting the data
x_train_mov, x_test_mov, y_train_mov, y_test_mov = train_test_split(x, y_mov, test_size=0.2, random_state=42)
x_train_win, x_test_win, y_train_win, y_test_win = train_test_split(x, y_win, test_size=0.2, random_state=42)

# Feature scaling
scaler = StandardScaler()

x_train_mov_scaled = scaler.fit_transform(x_train_mov)
x_test_mov_scaled = scaler.transform(x_test_mov)

x_train_win_scaled = scaler.fit_transform(x_train_mov)
x_test_win_scaled = scaler.transform(x_test_mov)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Define the mov model
model_mov = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(x_train_mov_scaled.shape[1],)),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1)  # Output layer for regression should have 1 unit without activation
])

# Compile the mov model
model_mov.compile(optimizer='adam',
              loss='mean_squared_error',
              metrics=['mean_absolute_error'])

In [ ]:
history_mov = model_mov.fit(x_train_mov_scaled, y_train_mov, validation_split=0.2, epochs=EPOCHS, batch_size=32, verbose=1)

In [ ]:
model_win = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(x_train_win_scaled.shape[1],)),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')  # Changed for binary classification
])

model_win.compile(optimizer='adam',
              loss='binary_crossentropy',  # Changed for binary classification
              metrics=['accuracy'])  # Common metric for classification

In [ ]:
history_win = model_win.fit(x_train_win_scaled, y_train_win, validation_split=0.2, epochs=EPOCHS, batch_size=32, verbose=1)

In [ ]:
# Evaluate the model
test_loss_mov, test_mae = model_mov.evaluate(x_test_mov, y_test_mov, verbose=1)
print('\nTest loss mov:', test_mae)

test_loss_win, test_accuracy = model_win.evaluate(x_test_win, y_test_win, verbose=1)
print(f'Test Loss win: {test_loss_win}, Test Accuracy: {test_accuracy}')


In [ ]:
def get_team_kenpom_by_id(teamID, season=SEASON):
    return kenpom.loc[(kenpom['Season'] == season) & (kenpom['TeamID'] == teamID)].iloc[0]

def get_team_kenpom_by_team_name(team_name, season=SEASON):
    team = teams_df.loc[teams_df['TeamNameSpelling'] == team_name.lower()]
    assert not team.empty, f"No team found with spelling {team_name}"
    teamID = team.iloc[0]['TeamID']
    
    return get_team_kenpom_by_id(teamID, season)

def get_team_features(team_kenpom):
    return [team_kenpom['adj_o'], team_kenpom['adj_d'], team_kenpom['adj_tempo'], team_kenpom['luck'], team_kenpom['sos_adj_o'], team_kenpom['sos_adj_d']]
#     return [team_kenpom[kenpom_columns]]
    
def is_number(value):
    return isinstance(value, (int, float, complex))

def mov_to_win_prob(mov, k=0.14):
    """
    Convert margin of victory to win probability.
    
    :param mov: Predicted margin of victory (positive for Team 1 win, negative for Team 2 win)
    :param k: Constant determining the steepness of the function
    :return: Probability of Team 1 winning
    """
    win_prob = 1 / (1 + np.exp(-k * mov))
    return win_prob

def predict_matchup(team1, team2, season=SEASON, location=0, label_type=LABEL_TYPE):
    if is_number(team1):
        team1_kenpom = get_team_kenpom_by_id(team1, season)
    else:
        team1_kenpom = get_team_kenpom_by_team_name(team1, season)
    if is_number(team2):
        team2_kenpom = get_team_kenpom_by_id(team2, season)
    else:
        team2_kenpom = get_team_kenpom_by_team_name(team2, season)
        
    
    team1_features = get_team_features(team1_kenpom)
    team2_features = get_team_features(team2_kenpom)
    
    column_names = ['t1_loc_num'] + team1_kenpom_columns + team2_kenpom_columns
    
    full_features = [location] + team1_features + team2_features
    full_features_flipped = [-location] + team2_features + team1_features
    
    full_features_scaled = scaler.transform(pd.DataFrame([full_features], columns=column_names))
    full_features_flipped_scaled = scaler.transform(pd.DataFrame([full_features_flipped], columns=column_names))

    if (label_type == LabelType.MARGIN_OF_VICTORY):
        predicted_mov = model_mov.predict(full_features_scaled, verbose=0)[0][0]
        predicted_mov_flipped = model_mov.predict(full_features_flipped_scaled, verbose=0)[0][0]
        final_predicted_mov = (predicted_mov - predicted_mov_flipped) / 2
        derived_win = mov_to_win_prob(final_predicted_mov)
        
        return derived_win

    elif (label_type == LabelType.WIN_PROBABILITY):
        predicted_win = model_win.predict(full_features_scaled, verbose=0)[0][0]
        predicted_win_flipped = model_win.predict(full_features_flipped_scaled, verbose=0)[0][0]
        final_predicted_win = (predicted_win + (1.0 - predicted_win_flipped)) / 2 

        return final_predicted_win

    return None

In [ ]:
print(predict_matchup(1222, 1188, SEASON, 0, LabelType.MARGIN_OF_VICTORY)) # houston vs siue
print(predict_matchup('houston', 'siue', SEASON, 0, LabelType.MARGIN_OF_VICTORY))
print(predict_matchup('duke', 'florida', SEASON, 0, LabelType.MARGIN_OF_VICTORY))
print(predict_matchup('florida', 'duke', SEASON, 0, LabelType.MARGIN_OF_VICTORY))
print(predict_matchup('ole miss', 'unc', SEASON, 0, LabelType.MARGIN_OF_VICTORY))
print(predict_matchup('unc', 'ole miss', SEASON, 0, LabelType.MARGIN_OF_VICTORY))

In [ ]:
print(predict_matchup(1222, 1188, SEASON, 0, LabelType.WIN_PROBABILITY)) # houston vs siue
print(predict_matchup('houston', 'siue', SEASON, 0, LabelType.WIN_PROBABILITY))
print(predict_matchup('duke', 'florida', SEASON, 0, LabelType.WIN_PROBABILITY))
print(predict_matchup('florida', 'duke', SEASON, 0, LabelType.WIN_PROBABILITY))
print(predict_matchup('ole miss', 'unc', SEASON, 0, LabelType.WIN_PROBABILITY))
print(predict_matchup('unc', 'ole miss', SEASON, 0, LabelType.WIN_PROBABILITY))

In [ ]:
def prepare_data(seeds):
    seed_dict = seeds.set_index('Seed')['TeamID'].to_dict()
    inverted_seed_dict = {value: key for key, value in seed_dict.items()}

    return seed_dict, inverted_seed_dict

# 2025
seeds = CSV['MNCAATourneySeeds']
seeds = seeds[seeds['Season'] == SEASON]

# display(seeds)

seed_dict, inverted_seed_dict = prepare_data(seeds)

print(seed_dict)
print(inverted_seed_dict)

In [ ]:
def build_win_probability_matrix(label_type=LABEL_TYPE):
    win_probability_matrix = {}
    for team_id in inverted_seed_dict.keys():
        win_probability_matrix[team_id] = {}
        for opponent_id in inverted_seed_dict.keys():
            if team_id == opponent_id:
                continue
            if opponent_id in win_probability_matrix and team_id in win_probability_matrix[opponent_id]:
                win_probability_matrix[team_id][opponent_id] = 1.0 - win_probability_matrix[opponent_id][team_id]
            else:
                win_probability_matrix[team_id][opponent_id] = predict_matchup(team_id, opponent_id, season=SEASON, location=0, label_type=LABEL_TYPE)
    return win_probability_matrix

# Convert float32 values to Python float
def convert_to_serializable(obj):
    if isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, np.float32):  # Convert np.float32 to float
        return float(obj)
    return obj

def write_to_json(win_probability_matrix, filename):
    serializable_data = convert_to_serializable(win_probability_matrix)

    # Write to JSON file
    with open(filename, "w") as json_file:
        json.dump(serializable_data, json_file)
    
    print(f'{filename} written successfully!')
    

In [ ]:
mov_matrix = build_win_probability_matrix(LabelType.MARGIN_OF_VICTORY)
write_to_json(mov_matrix, f'{SEASON}_mov_win_probability_matrix.json')

In [ ]:
win_matrix = build_win_probability_matrix(LabelType.WIN_PROBABILITY)
write_to_json(win_matrix, f'{SEASON}_win_win_probability_matrix.json')

In [ ]:
def build_slots(season=SEASON):
    slots = CSV["MNCAATourneySlots"]
    
    slots = slots[slots['Season'] == season]

    # Select early rounds where 'Slot' does NOT start with 'R'
    play_ins = slots[~slots['Slot'].str.startswith('R')]

    # Select remaining slots where 'Slot' starts with 'R'
    actual_slots = slots[slots['Slot'].str.startswith('R')]

    # Concatenate early rounds at the top
    sorted_slots = pd.concat([play_ins, actual_slots], ignore_index=True)

    return sorted_slots

slots = build_slots()
display(slots)

In [ ]:
def simulate(round_slots, seeds, inverted_seeds, win_probability_matrix):
    '''
    Simulates each round of the tournament.

    Parameters:
    - round_slots: DataFrame containing information on who is playing in each round.
    - seeds (dict): Dictionary mapping seed values to team IDs.
    - inverted_seeds (dict): Dictionary mapping team IDs to seed values.
    - wins (DF): DF that includes wins prediction per matchup.
    Returns:
    - list: List with winning team IDs for each match.
    - list: List with corresponding slot names for each match.
    '''
    winners = []
    slots = []

    for slot, strong, weak in zip(round_slots.Slot, round_slots.StrongSeed, round_slots.WeakSeed):
        team_1, team_2 = seeds[strong], seeds[weak]

        team_1_prob = win_probability_matrix[team_1][team_2]

        p = np.array([team_1_prob, 1 - team_1_prob])
        p /= p.sum()  # Normalize to ensure sum is exactly 1.0

        # winner = np.random.choice([team_1, team_2], p=[team_1_prob, 1 - team_1_prob])
        winner = np.random.choice([team_1, team_2], p=p)

        # Append the winner and corresponding slot to the lists
        winners.append(winner)
        slots.append(slot)

        seeds[slot] = winner

    return [inverted_seeds[w] for w in winners], slots


def run_simulation(seeds, round_slots, win_probability_matrix, num_brackets):
    '''
    Runs a simulation of bracket tournaments.

    Parameters:
    - seeds (pd.DataFrame): DataFrame containing seed information.
    - round_slots (pd.DataFrame): DataFrame containing information about the tournament rounds.
    - wins (DF): DF that includes wins prediction per matchup.
    - brackets (int): Number of brackets to simulate.
    Returns:
    - pd.DataFrame: DataFrame with simulation results.
    '''
    # Get relevant data for the simulation
    seed_dict, inverted_seed_dict = prepare_data(seeds)
    # Lists to store simulation results
    results = []
    bracket = []
    slots = []

    # Iterate through the specified number of brackets
    for b in tqdm(range(1, num_brackets + 1)):
        # Run single simulation
        r, s = simulate(round_slots, seed_dict, inverted_seed_dict, win_probability_matrix)

        # Update results
        results.extend(r)
        bracket.extend([b] * len(r))
        slots.extend(s)

    # Create final DataFrame
    result_df = pd.DataFrame({'Bracket': bracket, 'Slot': slots, 'Team': results})

    return result_df

In [ ]:
num_brackets = 100000

In [ ]:
mov_result = run_simulation(seeds, slots, mov_matrix, num_brackets)
mov_result.insert(0, 'Tournament', 'M')

display(mov_result)

In [ ]:
win_result = run_simulation(seeds, slots, win_matrix, num_brackets)
win_result.insert(0, 'Tournament', 'M')

display(win_result)

In [ ]:
def result_to_submission(result, filename):
    submission = result
    submission.reset_index(inplace=True, drop=True)
    submission.index.names = ['RowId']
    submission.to_csv(filename)

In [ ]:
result_to_submission(mov_result, f'{SEASON}_mov_my_submission.csv')
result_to_submission(win_result, f'{SEASON}_win_my_submission.csv')